# Qwen Chat-Template Validation
FP16-only validation of the pilot subsets. Expected ranges: MMLU 0.55-0.65 and GSM8K 0.55-0.65. Runtime is approximately 50 minutes on a T4.

In [ ]:
from pathlib import Path

roots = [p.parent for p in Path('/kaggle/input').glob('**/requirements.txt') if (p.parent / 'pilot_eval').is_dir()]
assert roots, 'Attach the repository Kaggle dataset before running this notebook.'
SOURCE = roots[0]
WORK = Path('/kaggle/working/compression-eval')
print('source:', SOURCE)

In [ ]:
import shutil, subprocess, sys

if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE, WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements.txt')], check=True)

In [ ]:
import yaml

source_config = WORK / 'configs/kaggle_qwen_public_quantized.yaml'
config = yaml.safe_load(source_config.read_text())
config['run_name'] = 'kaggle_qwen25_1p5b_fp16_chat_validation'
config['methods'] = []
for task in config['tasks']:
    task['prompt_style'] = 'chat'
    task['fewshot_style'] = 'inline'
validation_config = WORK / 'configs/kaggle_template_validation.yaml'
validation_config.write_text(yaml.safe_dump(config, sort_keys=False))
print(validation_config.read_text())

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pilot_eval.run', '--config', str(validation_config), '--only-method', 'fp16'],
    cwd=WORK, check=True,
)

In [ ]:
import json

run_dir = Path('/kaggle/working/results') / config['run_name']
accuracies = {}
counts = {}
prompt_styles = {}
for task in ('mmlu', 'gsm8k'):
    records = [json.loads(line) for line in (run_dir / f'fp16.{task}.jsonl').read_text().splitlines()]
    counts[task] = len(records)
    accuracies[task] = sum(record['correct'] for record in records) / len(records)
    prompt_styles[task] = sorted({record['metadata']['prompt_style'] for record in records})
    print(f'{task}: n={counts[task]}, accuracy={accuracies[task]:.3f}, prompt_styles={prompt_styles[task]}')

gate_passed = (
    all(0.55 <= accuracies[task] <= 0.65 for task in ('mmlu', 'gsm8k'))
    and all(prompt_styles[task] == ['chat'] for task in ('mmlu', 'gsm8k'))
)

In [ ]:
import shutil, tarfile
from datetime import datetime, timezone

summary = {
    'run_name': config['run_name'],
    'created_at': datetime.now(timezone.utc).isoformat(),
    'model_id': config['baseline']['model_id'],
    'dtype': config['baseline']['dtype'],
    'accuracies': accuracies,
    'counts': counts,
    'prompt_styles': prompt_styles,
    'expected_accuracy_range': [0.55, 0.65],
    'gate_passed': gate_passed,
}
(run_dir / 'validation_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
shutil.copy2(validation_config, run_dir / 'validation_config.yaml')
archive = Path('/kaggle/working/kaggle_qwen25_1p5b_fp16_chat_validation.tar.gz')
with tarfile.open(archive, 'w:gz') as tar:
    tar.add(run_dir, arcname=run_dir.name)

print(json.dumps(summary, indent=2))
print(f'preserved archive: {archive}')
print('GATE: PASS — pipeline validation is in range.' if gate_passed else 'GATE: STOP AND DEBUG — preserve these outputs and do not change the protocol.')